# Concurrency Sweeps for Generative AI Inference on SageMaker AI

## Why run a concurrency sweep?

When you're planning to serve a model in production, you need to answer a fundamental capacity question:

*How many concurrent requests can a single endpoint handle before latency becomes unacceptable?*

This matters because:

- *Capacity planning*: If your traffic peaks at 50 concurrent users, you need to know whether one instance handles it or you need five. That's a 5× cost difference.

- *SLA validation*: You may have a latency target (e.g., "p99 < 10s"). A concurrency sweep tells you exactly where that target breaks.

- *Cost optimization*: Over-provisioning wastes money; under-provisioning drops requests. The sweep gives you the data to right-size.

A concurrency sweep systematically benchmarks your endpoint at increasing load levels and measures throughput and latency at each level. The result is a curve that shows you where your endpoint saturates — the "knee" beyond which adding more traffic degrades the experience. This notebook demonstrates how to run concurrency sweeps on a SageMaker endpoint using the `create_ai_benchmark_job` API from [Amazon SageMaker AI Inference Recommendations](https://aws.amazon.com/blogs/machine-learning/amazon-sagemaker-ai-now-supports-optimized-generative-ai-inference-recommendations/).

## What you'll learn

1. Deploy **NVIDIA Nemotron-3 Nano 30B** (MoE model with only 3B active parameters) to a `ml.g7e.2xlarge` instance.
2. Run a **concurrency sweep** (powers of 4: 64, 256, 1024, ..) on the endpoint.
3. Visualize **throughput vs. latency** curves to identify the saturation point.
4. Use the **find-max-concurrency** search to automatically discover the highest concurrency that meets a user-defined SLA.

## Prerequisites

- An AWS account with SageMaker AI access in **us-east-2**
- An IAM execution role with SageMaker + S3 permissions
- Python 3.10+ with `boto3`, `sagemaker`, `pandas`, `matplotlib`
- Service quota for `ml.g7e.2xlarge` real-time endpoints
- This notebook was tested on SageMaker Studio with the **Data Science 3.0** kernel

## 1. Setup & Configuration

In [ ]:
!pip install --upgrade boto3 botocore --quiet

In [ ]:
import boto3
import json
import time
import uuid
import pandas as pd
import matplotlib.pyplot as plt
from sagemaker.core.helper.session_helper import Session, get_execution_role

role = get_execution_role()
sess = Session()
bucket = sess.default_bucket()
region = sess._region_name

sm_client = boto3.client(service_name="sagemaker")
sm_runtime = boto3.client(service_name="sagemaker-runtime")

S3_OUTPUT = f"s3://{bucket}/concurrency-sweeps/"

# Model
MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
TOKENIZER = MODEL_ID

# Instance types to compare: Add other instance types that are of interest
INSTANCES = {
    "g7e": "ml.g7e.2xlarge",
}

CONCURRENCY_LEVELS = [4**i for i in range(3,6)]  # Using powers of 4

print(f"Region:       {region}")
print(f"Model:        {MODEL_ID}")
print(f"Instances:    {list(INSTANCES.values())}")
print(f"Concurrency:  {CONCURRENCY_LEVELS}")

## 2. Deploy the Model to Endpoint

We deploy Nemotron-3 Nano using the [**native vLLM container**](https://aws.github.io/deep-learning-containers/reference/available_images/#vllm) for SageMaker AI. This container uses
`SM_VLLM_*` environment variables to configure the vLLM engine directly.

Key configuration:
- `SM_VLLM_MODEL` — the HuggingFace model ID to download and serve
- `SM_VLLM_MAX_MODEL_LEN` — maximum sequence length (input + output)
- `SM_VLLM_GPU_MEMORY_UTILIZATION` — fraction of GPU memory for KV cache

> **Note:** Deployment takes ~5–10 minutes per endpoint.

In [ ]:
# Native vLLM container image for SageMaker
VLLM_IMAGE = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.19.1-gpu-py312-cu129-ubuntu22.04-sagemaker"


def deploy_endpoint(instance_key: str, instance_type: str) -> str:
    """Deploy the model on the native vLLM container and return the endpoint name."""
    endpoint_name = f"nemotron-nano-{instance_key}-{uuid.uuid4().hex[:6]}"

    # vLLM serving configuration
    # These properties are tested for the Nemotron-3 Nano MoE/Mamba hybrid architecture.
    environment = {
        "SM_VLLM_MODEL": MODEL_ID,
        "SM_VLLM_ENFORCE_EAGER": "true",            # required for Mamba-Transformer hybrid
        "SM_VLLM_TENSOR_PARALLEL_SIZE": "1",        # single GPU
        "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.85",
        "SM_VLLM_MAX_MODEL_LEN": "10240",
        "SM_VLLM_ENABLE_PREFIX_CACHING": "true",
        "SM_VLLM_TRUST_REMOTE_CODE": "true",
    }

    # Create model
    model_name = endpoint_name
    sm_client.create_model(
        ModelName=model_name,
        PrimaryContainer={
            "Image": VLLM_IMAGE,
            "Environment": environment,
        },
        ExecutionRoleArn=role,
    )

    # Create endpoint config
    sm_client.create_endpoint_config(
        EndpointConfigName=endpoint_name,
        ProductionVariants=[
            {
                "VariantName": "AllTraffic",
                "ModelName": model_name,
                "InstanceType": instance_type,
                "InitialInstanceCount": 1,
                "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
                "ModelDataDownloadTimeoutInSeconds": 1800,
            }
        ],
    )

    # Create endpoint
    print(f"Deploying to {instance_type} as '{endpoint_name}'...")
    sm_client.create_endpoint(
        EndpointName=endpoint_name,
        EndpointConfigName=endpoint_name,
    )

    # Wait for InService
    print("  Waiting for endpoint to be InService...")
    waiter = sm_client.get_waiter("endpoint_in_service")
    waiter.wait(
        EndpointName=endpoint_name,
        WaiterConfig={"Delay": 30, "MaxAttempts": 60},
    )
    print(f"  ✓ {endpoint_name} is InService")
    return endpoint_name


# Deploy
endpoints = {}
for key, inst in INSTANCES.items():
    endpoints[key] = deploy_endpoint(key, inst)

print("\n--- Endpoints ready ---")
for k, v in endpoints.items():
    print(f"  {k}: {v}")

In [ ]:
# Quick sanity check — invoke each endpoint with a simple question
test_payload = {
    "messages": [
        {"role": "user", "content": "What is the capital of Colorado?"}
    ],
    "max_tokens": 100,
    "temperature": 0.1,
    "chat_template_kwargs": {"enable_thinking": False},
}

for key, ep_name in endpoints.items():
    print(f"Testing {INSTANCES[key]} ({ep_name})...")
    response = sm_runtime.invoke_endpoint(
        EndpointName=ep_name,
        ContentType="application/json",
        Body=json.dumps(test_payload),
    )
    result = json.loads(response["Body"].read().decode("utf-8"))
    print(f"  ✓ {result['choices'][0]['message']['content'].strip()}\n")

## 3. Run a Benchmark Job

To prevent repetition, this notebook will use the `run_benchmark` helper function that wraps two API calls and poll continuously until the job completes:

1. [**`CreateAIWorkloadConfig`**](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateAIWorkloadConfig.html): Defines *what* to benchmark (workload shape, concurrency, SLA).
2. [**`CreateAIBenchmarkJob`**](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateAIBenchmarkJob.html): Runs the benchmark against a live endpoint.

In [ ]:
def run_benchmark(endpoint_name: str, parameters: dict, prefix: str) -> dict:
    """
    Create a workload config + benchmark job, poll until done, return describe response.
    """
    suffix = uuid.uuid4().hex[:8]
    config_name = job_name = f"{prefix}-{suffix}"

    # Base workload profile: These will be the same for all tests in the sweep
    base = {
        "tokenizer": TOKENIZER,
        "streaming": True,
        "prompt_input_tokens_mean": 1024,
        "output_tokens_mean": 256,
    }

    spec = {
        "benchmark": {"type": "aiperf"},
        "parameters": {**base, **parameters},
    }

    # 1) Create workload config
    sm_client.create_ai_workload_config(
        AIWorkloadConfigName=config_name,
        AIWorkloadConfigs={"WorkloadSpec": {"Inline": json.dumps(spec)}},
    )

    # 2) Launch benchmark job
    max_runtime_s = 3600
    sm_client.create_ai_benchmark_job(
        AIBenchmarkJobName=job_name,
        AIWorkloadConfigIdentifier=config_name,
        RoleArn=role,
        BenchmarkTarget={"Endpoint": {"Identifier": endpoint_name}},
        OutputConfig={"S3OutputLocation": f"{S3_OUTPUT}{job_name}/"},
    )

    print(f"⏱ Started job '{job_name}' — polling...")
    poll_interval = 30
    for i in range(max_runtime_s // poll_interval + 2):
        time.sleep(poll_interval)
        resp = sm_client.describe_ai_benchmark_job(AIBenchmarkJobName=job_name)
        status = resp["AIBenchmarkJobStatus"]
        elapsed = (i + 1) * poll_interval
        print(f"  [{elapsed:>4}s] {status}")
        if status in ("Completed", "Failed", "Stopped"):
            if status == "Failed":
                print(f"  ❌ Reason: {resp.get('FailureReason')}")
            break

    resp["_job_name"] = job_name
    resp["_config_name"] = config_name
    return resp

## 4. Run Concurrency Sweeps

We pass a **list** of concurrency levels. The benchmark engine ([AIPerf](https://docs.nvidia.com/aiperf/welcome-to-ai-perf-documentation)) runs each level sequentially
within a single job and writes per-level results to S3.

Each sweep sends 1024 requests per concurrency level, enough for stable estimates while keeping costs low.

In [ ]:
sweep_params = {
    "concurrency": CONCURRENCY_LEVELS,
    "request_count": 1024,
}

results = {}
for key, ep_name in endpoints.items():
    print(f"\n{'='*60}")
    print(f"Running sweep on {INSTANCES[key]} ({ep_name})")
    print(f"{'='*60}")
    results[key] = run_benchmark(ep_name, sweep_params, prefix=f"sweep-{key}")
    print(f"\n✓ {key} sweep status: {results[key]['AIBenchmarkJobStatus']}")

## 5. Parse Results & Compare

Benchmark outputs are stored in S3 as a tarball. Each concurrency level gets its own subdirectory
with a `results.json` containing latency percentiles and throughput metrics.

Below we download, extract, and build a comparison DataFrame.

In [ ]:
import tarfile, io

s3_client = boto3.client("s3")

def parse_sweep_results(job_name: str) -> pd.DataFrame:
    """Download benchmark output from S3 and extract per-concurrency metrics."""
    prefix = f"concurrency-sweeps/{job_name}/"

    # Find tarball (path includes an unpredictable subdirectory)
    tar_key = next(
        (obj["Key"]
         for page in s3_client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix)
         for obj in page.get("Contents", [])
         if obj["Key"].endswith("output.tar.gz")),
        None,
    )
    assert tar_key, f"No output.tar.gz under s3://{bucket}/{prefix}"

    tar_bytes = s3_client.get_object(Bucket=bucket, Key=tar_key)["Body"].read()

    with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode="r:gz") as tar:
        sweep = json.loads(tar.extractfile("sweep_aggregate/profile_export_aiperf_sweep.json").read())

    return pd.DataFrame([
        {
            "concurrency": c["parameters"]["concurrency"],
            "throughput_rps": c["metrics"]["request_throughput"]["mean"],
            "latency_p50_ms": c["metrics"]["request_latency"]["p50"],
            "latency_p99_ms": c["metrics"]["request_latency"]["p99"],
            "ttft_p50_ms": c["metrics"]["time_to_first_token"]["p50"],
            "ttft_p99_ms": c["metrics"]["time_to_first_token"]["p99"],
        }
        for c in sweep["per_combination_metrics"]
    ]).sort_values("concurrency").reset_index(drop=True)


# Parse results for each instance type in the sweep
dfs = {}
for key in endpoints:
    job_name = results[key]["_job_name"]
    dfs[key] = parse_sweep_results(job_name)
    print(f"\n--- {INSTANCES[key]} ---")
    display(dfs[key])

In [ ]:
# --- Comparison plots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {"g7e": "firebrick"}
labels = {"g7e": "g7e.2xlarge (L40S)"}

# Plot 1: Throughput vs Concurrency
ax = axes[0]
for key, df in dfs.items():
    ax.plot(df["concurrency"], df["throughput_rps"],
            marker="o", color=colors[key], label=labels[key])
ax.set_xscale("log", base=4)
ax.set_xlabel("Concurrency")
ax.set_ylabel("Throughput (tokens/sec)")
ax.set_title("Throughput vs. Concurrency")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: p99 E2E Latency vs Concurrency
ax = axes[1]
for key, df in dfs.items():
    ax.plot(df["concurrency"], df["latency_p99_ms"],
            marker="s", color=colors[key], label=labels[key])
ax.set_xscale("log", base=4)
ax.set_xlabel("Concurrency")
ax.set_ylabel("p99 End-to-End Latency (ms)")
ax.set_title("p99 Latency vs. Concurrency")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("concurrency_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n💡 The 'knee' of the latency curve shows where the endpoint saturates —")
print("   adding more concurrent requests no longer improves throughput, only hurts latency.")

## 6. Find-Max-Concurrency Search

Instead of manually picking concurrency levels, you can let the benchmark engine **search** for the highest concurrency that meets a latency SLA. This uses the `max-concurrency-under-sla` recipe, which accepts one or more of the following SLA thresholds.

| Parameter | Meaning | Fixed statistic | Requires streaming? |
|---|---|---|---|
| `ttft_sla_ms` | max Time To First Token (ms) | p95 | yes |
| `tpot_sla_ms` | max Time Per Output Token / inter-token latency (ms) | p95 | yes |
| `e2e_sla_ms` | max end-to-end request latency (ms) | p99 | no |
| `error_rate_sla` | max fraction of failed requests (e.g. `0.01` = 1%) | avg | no |

> ⚠️ **Note:** The statistic used for each SLA is **fixed by the recipe** — you cannot override it with `search_stat`. That flag is mutually exclusive with `search_recipe`. If you need a custom statistic (e.g., p50 instead of p99 for e2e), you would build a custom `search_space` search instead of using the named recipe.

**Tunable knobs** you *can* pass alongside the recipe:
- `concurrency_min` / `concurrency_max` — bound the search range (default: 1–1000)
- `search_max_iterations` — cap iterations to control cost/time
- `search_initial_points` — seed the search to converge faster
- `request_count` — number of requests per concurrency probe

Below we search for the max concurrency where the **p99 end-to-end latency stays under 50 seconds**.

In [ ]:
search_params = {
    "search_recipe": "max-concurrency-under-sla",
    "e2e_sla_ms": 50000,              # end-to-end latency ceiling (ms), evaluated at p99
    "concurrency_min": 16,            # optional; search range lower bound; default 1
    "concurrency_max": 4096,          # optional; search range upper bound; default 1000
    "search_max_iterations": 10,      # optional; cap iterations to bound cost
    "search_initial_points": 512,    # optional; initial guess to speed convergence
    "request_count": 1024,            # optional; number of requests per concurrency probe
}

print("Searching for max concurrency under 50s p99 E2E latency...")
search_result = run_benchmark(
    endpoints["g7e"],
    search_params,
    prefix="findmax",
)
print(f"\nSearch job status: {search_result['AIBenchmarkJobStatus']}")

### Interpreting the search results

The search output includes a `search_history.json` showing each iteration's concurrency guess
and whether it passed the SLA. The final answer is the **highest concurrency that passed all SLA gates**.

This is especially powerful when you:
- Don't know the right concurrency range ahead of time
- Want to automate capacity planning across model versions
- Need to combine multiple SLAs (e.g., TTFT < 500ms AND e2e < 5s)

In [ ]:
import tarfile, io

# Download the search job output
job_name = search_result["_job_name"]
prefix = f"concurrency-sweeps/{job_name}/"

tar_key = next(
    (obj["Key"]
     for page in s3_client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix)
     for obj in page.get("Contents", [])
     if obj["Key"].endswith("output.tar.gz")),
    None,
)
assert tar_key, f"No output.tar.gz under s3://{bucket}/{prefix}"

tar_bytes = s3_client.get_object(Bucket=bucket, Key=tar_key)["Body"].read()

# Parse search_history.json
with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode="r:gz") as tar:
    search_data = json.loads(tar.extractfile("search_history.json").read())

# Build summary table
search_df = pd.DataFrame([
    {
        "iteration": it["iteration_idx"],
        "concurrency": it["variation_values"]["phases.profiling.concurrency"],
        "throughput_tps": it["objective_values"][0] if it["objective_values"] else None,
        "passed_sla": "✓" if it["feasible"] else "✗",
    }
    for it in search_data["iterations"]
])

sla = search_data["config"]["sla_filters"][0]
print(f"SLA: {sla['stat']} {sla['metric_tag']} < {sla['threshold']:.0f} ms")
print(f"Search range: [{search_data['config']['search_space'][0]['lo']:.0f}, "
      f"{search_data['config']['search_space'][0]['hi']:.0f}]")
print(f"Planner: {search_data['config']['planner']}")
print(f"Iterations: {len(search_data['iterations'])}\n")
display(search_df)

# Winner — highest concurrency that passed
passed = search_df[search_df["passed_sla"] == "✓"]
if not passed.empty:
    winner = passed.loc[passed["concurrency"].idxmax()]
    print(f"\n🏆 Max concurrency under SLA: {int(winner['concurrency'])}")
    print(f"   Throughput at that level: {winner['throughput_tps']:.1f} tokens/sec")
else:
    print("\n⚠️ No concurrency level passed the SLA!")

## 7. Combining SLAs

You can also pass several thresholds at the same time, and the winning concurrency must satisfy **all** of them. Below we have an example that controls *both* the p99 end-to-end latency and the p95 time to first token.

> For this test, we will temporarily disable the `error_rate_sla` condition because AIPerf v0.12.0 does not emmit `request_error_rate` for runs with zero errors, which make every probe in a feasible run appear infeasible.

In [ ]:
search_params = {
    "search_recipe": "max-concurrency-under-sla",
    # Winner concurrency must satisfy BOTH e2e and ttft latencies
    "e2e_sla_ms": 50000,              # end-to-end latency ceiling (ms), evaluated at p99
    "ttft_sla_ms": 1500,              # time-to-first-token latency ceiling (ms); evaluated at p95
    "concurrency_min": 16,            # optional; search range lower bound; default 1
    "concurrency_max": 4096,          # optional; search range upper bound; default 1000
    "search_max_iterations": 10,      # optional; cap iterations to bound cost
    "search_initial_points": 512,    # optional; initial guess to speed convergence
    "request_count": 1024,            # optional; number of requests per concurrency probe
}

print("Searching for max concurrency under 50s p99 E2E latency and 1.5s o95 TTFT latency ...")
search_result = run_benchmark(
    endpoints["g7e"],
    search_params,
    prefix="findmax",
)
print(f"\nSearch job status: {search_result['AIBenchmarkJobStatus']}")

## 8. Cleanup

> 🚨 **Don't skip this step!** Your endpoint bills by the hour whether you're sending traffic or not. Run the cell below to delete it.

In [ ]:
for key, ep_name in endpoints.items():
    print(f"Deleting endpoint: {ep_name}")
    sm_client.delete_endpoint(EndpointName=ep_name)
    sm_client.delete_endpoint_config(EndpointConfigName=ep_name)
    sm_client.delete_model(ModelName=ep_name)

print("\n✓ All endpoints and models deleted.")
print("  Don't forget to delete the S3 benchmark outputs if no longer needed.")